# SPEED-EXP-006B — VS13 Wide-Subset Speed Calibration / Training

Amaç: VS13 veri setinde yalnız 3 araçlık sanity check yerine daha geniş araç/video subset'i ile mevcut tek kamera hız adayını kalibre etmek ve hafif tabular calibration modellerini karşılaştırmak.

Bu notebook iki seviyede çalışır:

1. **Kalibrasyon/tuning:** `YOLO + ByteTrack + bbox-geometry speed candidate` çıktısına `global_alpha`, `linear_raw` ve `huber_raw` kalibrasyonu uygular.
2. **Hafif tabular eğitim:** En iyi parametre adayları üzerinde `ridge_features`, `huber_features`, `random_forest_features` ve `gradient_boosting_features` regressor'larını dener.

Bu hâlâ yeni bir end-to-end neural speed modeli değildir. Eğitilen şey görüntü backbone'u değil, known-speed videolardan öğrenilen hız kalibrasyon katmanıdır.

Neden FTR ile uyumlu?

- FTR `results.json` içinde hız alanı zorunlu değildir; bu deney FTR ana teslimini bloklamaz.
- Hız sonucu yalnız `support/evidence` veya `calibrated approximate candidate` olarak raporlanır.
- Başarı değerlendirmesi frame-level leakage ile değil, **leave-one-vehicle-out** cross-validation ile yapılır.

VS13 kaynak notu: VS13 sayfası 13 araç ve 400 MP4 video içerdiğini, araç hızlarının cruise control ile sabit tutulduğunu ve video+annotation paketlerinin ayrı zip olarak indirilebildiğini belirtir: https://slobodan.ucg.ac.me/science/vs13/


In [ ]:
# Cell 1 — Install dependencies
# Colab runtime: GPU recommended for YOLO tracking. L4/T4 is enough; A100 only shortens tracking time.

!pip -q install ultralytics pandas matplotlib tqdm opencv-python scikit-learn "lap>=0.5.12"


In [ ]:
# Cell 2 — Imports and configuration
from __future__ import annotations

import csv
import hashlib
import json
import math
import os
import random
import re
import shutil
import statistics
import subprocess
import time
import zipfile
from collections import Counter, defaultdict
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import torch
from ultralytics import YOLO
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import HuberRegressor, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

try:
    from google.colab import drive
    IN_COLAB = True
except Exception:
    IN_COLAB = False

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

EXPERIMENT_ID = 'SPEED-EXP-006B-VS13-WIDE-SUBSET-CALIBRATION'
PROJECT_DRIVE_ROOT = Path('/content/drive/MyDrive/anomali-road-safety-ai')
LOCAL_WORK_ROOT = Path('/content/anomali-road-safety-ai-speed-exp-006b')

MOUNT_DRIVE = True
USE_DRIVE_CACHE = True
FORCE_REDOWNLOAD = False
# Set True only if you want to rebuild the local extracted subset in the current runtime.
FORCE_REEXTRACT = False

# Wide subset. Start with 10-12 for a practical run; set None to use every MP4 in selected packages.
MAX_VIDEOS_PER_VEHICLE = 12
FRAME_STRIDE = 1
TRACKER = 'bytetrack.yaml'
IMG_SIZE = 960
DETECT_CONF = 0.35
DEVICE = 0 if torch.cuda.is_available() else 'cpu'

# If you copied a fine-tuned checkpoint to Drive, set it here.
# Otherwise the notebook uses Ultralytics yolo11n.pt from the package/cache.
DRIVE_MODEL_CANDIDATES = [
    PROJECT_DRIVE_ROOT / 'models/checkpoints/vehicle_detection/VD-EXP-002-GENERAL-YOLO11N-best.pt',
    PROJECT_DRIVE_ROOT / 'yolo11n.pt',
]
FALLBACK_MODEL = 'yolo11n.pt'

VS13_BASE = 'https://slobodan.ucg.ac.me/science/vs13/video+annotations'
# Important: GT speed is not package-level. It is parsed from each video filename,
# e.g. RenaultCaptur_66.MP4 -> 66 km/h.
# Use all 13 VS13 video+annotation packages by default. Set a shorter list for a faster run.
SELECTED_VEHICLES = [
    'CitroenC4Picasso',
    'KiaSportage',
    'Mazda3',
    'MercedesAMG550',
    'MercedesGLA',
    'NissanQashqai',
    'OpelInsignia',
    'Peugeot208',
    'Peugeot3008',
    'Peugeot307',
    'RenaultCaptur',
    'RenaultScenic',
    'VWPassat',
]

VS13_PACKAGES = {
    'CitroenC4Picasso': {'url': f'{VS13_BASE}/CitroenC4Picasso.zip'},
    'KiaSportage': {'url': f'{VS13_BASE}/KiaSportage.zip'},
    'Mazda3': {'url': f'{VS13_BASE}/Mazda3.zip'},
    'MercedesAMG550': {'url': f'{VS13_BASE}/MercedesAMG550.zip'},
    'MercedesGLA': {'url': f'{VS13_BASE}/MercedesGLA.zip'},
    'NissanQashqai': {'url': f'{VS13_BASE}/NissanQashqai.zip'},
    'OpelInsignia': {'url': f'{VS13_BASE}/OpelInsignia.zip'},
    'Peugeot208': {'url': f'{VS13_BASE}/Peugeot208.zip'},
    'Peugeot3008': {'url': f'{VS13_BASE}/Peugeot3008.zip'},
    'Peugeot307': {'url': f'{VS13_BASE}/Peugeot307.zip'},
    'RenaultCaptur': {'url': f'{VS13_BASE}/RenaultCaptur.zip'},
    'RenaultScenic': {'url': f'{VS13_BASE}/RenaultScenic.zip'},
    'VWPassat': {'url': f'{VS13_BASE}/VWPassat.zip'},
}
VS13_PACKAGES = {name: VS13_PACKAGES[name] for name in SELECTED_VEHICLES}
DESCRIPTION_URL = f'{VS13_BASE}/%20Description%20(video%2Bannotations).pdf'

# First-stage grid. Keep this broad enough to test segment stability but bounded enough for Colab.
PARAM_GRID = {
    'horizontal_fov_deg': [55.0, 60.0, 65.0, 70.0, 75.0],
    'vehicle_height_m': [1.40, 1.50, 1.60, 1.70],
    'moving_average_window': [9, 15, 25, 35],
    'max_segment_speed_kmh': [140.0, 180.0, 220.0],
    'segment_trim_fraction': [0.0, 0.10, 0.20],
    'min_bbox_height_ratio': [0.0, 0.06, 0.10],
}

FIRST_STAGE_METHODS = ['global_alpha', 'linear_raw', 'huber_raw']
ENABLE_SECOND_STAGE_TABULAR = True
SECOND_STAGE_TOP_PARAM_CONFIGS = 20
SECOND_STAGE_METHODS = [
    'ridge_features',
    'huber_features',
    'random_forest_features',
    'gradient_boosting_features',
]

FEATURE_COLUMNS = [
    'pred_raw_kmh',
    'candidate_confidence',
    'candidate_speed_cv',
    'selected_observation_count',
    'selected_mean_confidence',
    'valid_segment_count',
    'candidate_valid_ratio',
]


print('IN_COLAB:', IN_COLAB)
print('DEVICE:', DEVICE)
print('EXPERIMENT_ID:', EXPERIMENT_ID)


In [ ]:
# Cell 3 — Mount Drive and create folders
if IN_COLAB and MOUNT_DRIVE:
    drive.mount('/content/drive')

RAW_DIR = (PROJECT_DRIVE_ROOT if USE_DRIVE_CACHE else LOCAL_WORK_ROOT) / 'data/external/vs13/raw'
EXTRACT_DIR = LOCAL_WORK_ROOT / 'data/external/vs13/extracted_subset'
ARTIFACT_DIR = PROJECT_DRIVE_ROOT / 'models/benchmarks/artifacts/speed/SPEED-EXP-006-VS13-known-speed-calibration'
RUN_DIR = PROJECT_DRIVE_ROOT / 'runs/speed/SPEED-EXP-006-VS13-known-speed-calibration'
PLOT_DIR = RUN_DIR / 'plots'

for d in [RAW_DIR, EXTRACT_DIR, ARTIFACT_DIR, RUN_DIR, PLOT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('RAW_DIR:', RAW_DIR)
print('EXTRACT_DIR:', EXTRACT_DIR)
print('ARTIFACT_DIR:', ARTIFACT_DIR)
print('RUN_DIR:', RUN_DIR)

In [ ]:
# Cell 4 — Download VS13 packages to Drive/local cache

def download_file(url: str, target: Path, min_size_mb: float = 1.0, force: bool = False):
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists() and target.stat().st_size > min_size_mb * 1024 * 1024 and not force:
        print('exists:', target, 'size MB:', round(target.stat().st_size / 1024 / 1024, 2))
        return target
    part = target.with_suffix(target.suffix + '.part')
    if part.exists():
        part.unlink()
    cmd = ['wget', '-O', str(part), '--tries=5', '--timeout=60', '--continue', url]
    print('+', ' '.join(cmd))
    subprocess.run(cmd, check=True)
    if part.stat().st_size < min_size_mb * 1024 * 1024:
        raise RuntimeError(f'Download too small: {part} size={part.stat().st_size}')
    part.rename(target)
    print('downloaded:', target, 'size MB:', round(target.stat().st_size / 1024 / 1024, 2))
    return target

# Description PDF is small; useful for citation/review.
download_file(DESCRIPTION_URL, RAW_DIR / 'Description_video_annotations.pdf', min_size_mb=0.05, force=False)

zip_paths = {}
for vehicle, meta in VS13_PACKAGES.items():
    target = RAW_DIR / f'{vehicle}.zip'
    zip_paths[vehicle] = download_file(meta['url'], target, min_size_mb=50.0, force=FORCE_REDOWNLOAD)

zip_paths

In [ ]:
# Cell 5 — Inspect zips and extract the selected wide subset of MP4 + annotation files

def list_zip_preview(zip_path: Path, limit: int = 30):
    with zipfile.ZipFile(zip_path) as zf:
        names = zf.namelist()
    print('\nZIP:', zip_path.name, 'files:', len(names))
    for name in names[:limit]:
        print(' ', name)
    return names


def video_files_under(root: Path):
    return sorted([p for p in root.rglob('*') if p.is_file() and p.suffix.lower() == '.mp4'])


def parse_vs13_speed_from_name(name: str):
    # VS13 files are named like RenaultCaptur_66.MP4. The suffix is km/h.
    stem = Path(name).stem
    match = re.search(r'_(\d+(?:\.\d+)?)$', stem)
    return float(match.group(1)) if match else None


def balanced_video_selection(mp4_names: list[str], max_videos):
    # Sort by numeric speed and sample evenly so the run covers low/mid/high speeds.
    items = []
    for name in mp4_names:
        speed = parse_vs13_speed_from_name(name)
        if speed is not None:
            items.append((speed, name))
    if max_videos is None:
        return sorted(mp4_names)
    if not items:
        return sorted(mp4_names)[:max_videos]
    items = sorted(items, key=lambda item: (item[0], item[1]))
    if len(items) <= max_videos:
        return [name for _speed, name in items]
    idxs = np.linspace(0, len(items) - 1, max_videos).round().astype(int).tolist()
    # Preserve uniqueness if rounding duplicates an index.
    selected = []
    for idx in idxs:
        if idx not in selected:
            selected.append(idx)
    cursor = 0
    while len(selected) < max_videos and cursor < len(items):
        if cursor not in selected:
            selected.append(cursor)
        cursor += 1
    return [items[idx][1] for idx in sorted(selected)]

for zp in zip_paths.values():
    list_zip_preview(zp, limit=12)


def extract_subset_for_vehicle(vehicle: str, zip_path: Path, max_videos):
    out_dir = EXTRACT_DIR / vehicle
    if out_dir.exists() and not FORCE_REEXTRACT:
        existing = video_files_under(out_dir)
        if max_videos is None and existing:
            print(vehicle, 'already extracted videos:', len(existing))
            return existing
        if max_videos is not None and len(existing) >= max_videos:
            print(vehicle, 'already extracted videos:', len(existing))
            return existing[:max_videos]
    if FORCE_REEXTRACT and out_dir.exists():
        shutil.rmtree(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_path) as zf:
        names = zf.namelist()
        mp4s = sorted([n for n in names if n.lower().endswith('.mp4')])
        selected_mp4s = balanced_video_selection(mp4s, max_videos)
        selected_stems = {Path(n).stem for n in selected_mp4s}
        selected = []
        for n in names:
            stem = Path(n).stem
            suffix = Path(n).suffix.lower()
            if n in selected_mp4s or (stem in selected_stems and suffix in {'.txt', '.csv', '.json'}):
                selected.append(n)
        print(vehicle, 'selected mp4:', len(selected_mp4s), 'selected files:', len(selected))
        print(' selected speeds:', [parse_vs13_speed_from_name(n) for n in selected_mp4s])
        for n in tqdm(selected, desc=f'extract {vehicle}'):
            zf.extract(n, out_dir)
    extracted = video_files_under(out_dir)
    if not extracted:
        raise RuntimeError(f'No MP4 files found after extracting {vehicle}. Check zip layout and suffix handling.')
    return extracted if max_videos is None else extracted[:max_videos]

video_paths = []
for vehicle, zp in zip_paths.items():
    video_paths.extend(extract_subset_for_vehicle(vehicle, zp, MAX_VIDEOS_PER_VEHICLE))

print('Extracted videos:', len(video_paths))
for p in video_paths[:30]:
    print(p, 'gt_speed_kmh=', parse_vs13_speed_from_name(p.name))


In [ ]:
# Cell 6 — Build VS13 manifest

def infer_vehicle_from_path(path: Path):
    text = str(path).lower()
    for vehicle in VS13_PACKAGES:
        if vehicle.lower() in text:
            return vehicle
    return path.parts[-2]

records = []
for p in video_files_under(EXTRACT_DIR):
    vehicle = infer_vehicle_from_path(p)
    meta = VS13_PACKAGES.get(vehicle)
    gt_speed = parse_vs13_speed_from_name(p.name)
    if not meta or gt_speed is None:
        continue
    records.append({
        'video_path': str(p),
        'video_name': p.name,
        'vehicle': vehicle,
        'gt_speed_kmh': float(gt_speed),
        'split': meta['split'],
    })

if not records:
    raise RuntimeError(
        'No VS13 videos found for manifest. Run the extraction cell again. '
        'The notebook expects .MP4/.mp4 files and parses GT speed from names like RenaultCaptur_66.MP4.'
    )

manifest_df = pd.DataFrame(records).sort_values(['split', 'vehicle', 'gt_speed_kmh', 'video_name']).reset_index(drop=True)
manifest_path = ARTIFACT_DIR / 'vs13_subset_manifest.csv'
manifest_df.to_csv(manifest_path, index=False)
print('manifest:', manifest_path, manifest_df.shape)
display(manifest_df.groupby(['split', 'vehicle']).agg(video_count=('video_name', 'count'), min_speed=('gt_speed_kmh', 'min'), max_speed=('gt_speed_kmh', 'max')).reset_index())
display(manifest_df.head(40))


In [ ]:
# Cell 7 — Resolve model checkpoint

def resolve_model_path():
    for p in DRIVE_MODEL_CANDIDATES:
        if p.exists():
            print('Using Drive model:', p)
            return str(p)
    print('Using fallback model:', FALLBACK_MODEL)
    return FALLBACK_MODEL

MODEL_PATH = resolve_model_path()
model = YOLO(MODEL_PATH)
print('model names:', model.names)

In [ ]:
# Cell 8 — Speed candidate functions

VEHICLE_CLASS_NAMES = {'car', 'motorcycle', 'bus', 'truck'}

def now_utc():
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat().replace('+00:00', 'Z')


def vehicle_class_ids(model):
    names = getattr(model, 'names', {}) or {}
    ids = [int(i) for i, name in names.items() if str(name) in VEHICLE_CLASS_NAMES]
    if not ids:
        # COCO fallback: car/motorcycle/bus/truck ids are usually 2/3/5/7.
        ids = [2, 3, 5, 7]
    return ids


def focal_lengths(width, height, horizontal_fov_deg):
    hfov = math.radians(horizontal_fov_deg)
    fx = width / (2.0 * math.tan(hfov / 2.0))
    vfov = 2.0 * math.atan(math.tan(hfov / 2.0) * (height / max(width, 1)))
    fy = height / (2.0 * math.tan(vfov / 2.0))
    return fx, fy


def moving_average(values, window):
    out = []
    for idx in range(len(values)):
        sample = [v for v in values[max(0, idx-window+1):idx+1] if v is not None and math.isfinite(v)]
        out.append(float(statistics.fmean(sample)) if sample else None)
    return out


def robust_mean(values):
    vals = np.array([v for v in values if v is not None and math.isfinite(v)], dtype=float)
    if len(vals) == 0:
        return None
    if len(vals) >= 8:
        lo, hi = np.percentile(vals, [10, 90])
        vals = vals[(vals >= lo) & (vals <= hi)]
    return float(np.mean(vals)) if len(vals) else None


def extract_main_track(video_path: Path, frame_stride=1):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f'Could not open video: {video_path}')
    fps = float(cap.get(cv2.CAP_PROP_FPS) or 30.0)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH) or 0)
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) or 0)
    class_ids = vehicle_class_ids(model)
    model_names = getattr(model, 'names', {}) or {}

    obs_by_track = defaultdict(list)
    frame_id = 0
    start = time.perf_counter()
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        frame_id += 1
        if frame_stride > 1 and (frame_id - 1) % frame_stride != 0:
            continue
        results = model.track(
            frame,
            persist=True,
            tracker=TRACKER,
            classes=class_ids,
            conf=DETECT_CONF,
            imgsz=IMG_SIZE,
            device=DEVICE,
            verbose=False,
        )
        result = results[0]
        if result.boxes is None or result.boxes.id is None:
            continue
        ids = result.boxes.id.int().cpu().tolist()
        clss = result.boxes.cls.int().cpu().tolist()
        confs = result.boxes.conf.cpu().tolist()
        boxes = result.boxes.xyxy.cpu().tolist()
        for tid, cls_id, conf, box in zip(ids, clss, confs, boxes):
            x1, y1, x2, y2 = map(float, box)
            bw, bh = max(0, x2-x1), max(0, y2-y1)
            if bw <= 1 or bh <= 1:
                continue
            obs_by_track[int(tid)].append({
                'frame_id': frame_id,
                'time_s': frame_id / fps,
                'track_id': int(tid),
                'class_name': str(model_names.get(int(cls_id), cls_id)),
                'confidence': float(conf),
                'bbox_width_px': bw,
                'bbox_height_px': bh,
                'bbox_area_px': bw * bh,
                'bottom_center_x': (x1 + x2) / 2,
                'bottom_center_y': y2,
            })
    cap.release()

    tracks = []
    for tid, obs in obs_by_track.items():
        if len(obs) < 8:
            continue
        areas = [o['bbox_area_px'] for o in obs]
        confs = [o['confidence'] for o in obs]
        tracks.append({
            'track_id': tid,
            'observation_count': len(obs),
            'mean_area': float(np.mean(areas)),
            'median_area': float(np.median(areas)),
            'mean_confidence': float(np.mean(confs)),
            'obs': obs,
        })
    if not tracks:
        return {
            'status': 'failed',
            'failure_reason': 'no_track',
            'fps': fps,
            'frame_count': frame_count,
            'resolution': [width, height],
            'runtime_s': time.perf_counter() - start,
        }
    # Single-car pass-by videos: prefer long and large track.
    best = max(tracks, key=lambda t: (t['observation_count'], t['median_area'], t['mean_confidence']))
    return {
        'status': 'ok',
        'failure_reason': None,
        'fps': fps,
        'frame_count': frame_count,
        'resolution': [width, height],
        'runtime_s': time.perf_counter() - start,
        'track_count': len(tracks),
        'selected_track_id': best['track_id'],
        'selected_observation_count': best['observation_count'],
        'selected_mean_confidence': best['mean_confidence'],
        'observations': best['obs'],
    }


def compute_speed_candidate(track_result, params):
    if track_result.get('status') != 'ok':
        return {'status': 'failed', 'failure_reason': track_result.get('failure_reason')}
    obs = sorted(track_result['observations'], key=lambda x: x['frame_id'])
    width, height = track_result['resolution']
    fx, fy = focal_lengths(width, height, params['horizontal_fov_deg'])
    cx = width / 2.0
    vehicle_height_m = params['vehicle_height_m']
    rows = []
    prev = None
    raw_speeds = []
    for o in obs:
        bbox_h = max(o['bbox_height_px'], 1.0)
        z_m = fy * vehicle_height_m / bbox_h
        x_m = (o['bottom_center_x'] - cx) * z_m / fx
        row = dict(o)
        row['z_m'] = z_m
        row['x_m'] = x_m
        row['segment_speed_kmh_raw'] = None
        row['segment_valid'] = None
        row['segment_failure_reason'] = None
        if prev is not None:
            dt = max(1e-6, row['time_s'] - prev['time_s'])
            dx = row['x_m'] - prev['x_m']
            dz = row['z_m'] - prev['z_m']
            speed = math.sqrt(dx*dx + dz*dz) / dt * 3.6
            valid = True
            reason = None
            if not math.isfinite(speed) or speed > params['max_segment_speed_kmh']:
                valid = False
                reason = 'speed_outlier_gate'
            height_ratio = max(row['bbox_height_px'], prev['bbox_height_px']) / max(1.0, min(row['bbox_height_px'], prev['bbox_height_px']))
            if height_ratio > 1.25:
                valid = False
                reason = 'bbox_height_jump'
            row['segment_speed_kmh_raw'] = speed if valid else None
            row['segment_valid'] = valid
            row['segment_failure_reason'] = reason
            raw_speeds.append(row['segment_speed_kmh_raw'])
        rows.append(row)
        prev = row
    ma = moving_average(raw_speeds, params['moving_average_window'])
    # raw_speeds has len rows-1; align by skipping first row.
    for row, ma_value in zip(rows[1:], ma):
        row['segment_speed_kmh_moving_avg'] = ma_value
    if rows:
        rows[0]['segment_speed_kmh_moving_avg'] = None
    candidate_rows = rows[1:]
    trim = float(params.get('segment_trim_fraction', 0.0) or 0.0)
    if candidate_rows and trim > 0:
        start_idx = int(len(candidate_rows) * trim)
        end_idx = int(len(candidate_rows) * (1.0 - trim))
        candidate_rows = candidate_rows[start_idx:max(start_idx + 1, end_idx)]
    min_bbox_height_ratio = float(params.get('min_bbox_height_ratio', 0.0) or 0.0)
    if min_bbox_height_ratio > 0:
        min_h = height * min_bbox_height_ratio
        candidate_rows = [r for r in candidate_rows if r.get('bbox_height_px', 0.0) >= min_h]
    ma_values = [r.get('segment_speed_kmh_moving_avg') for r in candidate_rows]
    raw_values = [r.get('segment_speed_kmh_raw') for r in candidate_rows]
    estimate = robust_mean(ma_values)
    valid_count = sum(1 for v in raw_values if v is not None)
    speed_std = float(np.std([v for v in ma_values if v is not None])) if any(v is not None for v in ma_values) else None
    speed_mean = float(np.mean([v for v in ma_values if v is not None])) if any(v is not None for v in ma_values) else None
    speed_cv = (speed_std / speed_mean) if speed_mean and speed_mean > 0 else None
    coverage_quality = min(len(obs) / 180.0, 1.0)
    valid_ratio = valid_count / max(len(obs) - 1, 1)
    valid_ratio = max(0.0, min(1.0, valid_ratio))
    cv_quality = 0.0 if speed_cv is None else max(0.0, min(1.0, 1.0 - speed_cv / 0.35))
    mean_det_conf = float(np.mean([o.get('confidence', 0.0) for o in obs])) if obs else 0.0
    det_conf_quality = max(0.0, min(1.0, mean_det_conf / 0.80))
    estimate_bonus = 0.10 if estimate is not None else 0.0
    confidence = (
        0.08
        + 0.18 * coverage_quality
        + 0.22 * valid_ratio
        + 0.30 * cv_quality
        + 0.12 * det_conf_quality
        + estimate_bonus
    )
    confidence = round(max(0.0, min(confidence, 0.90)), 4)
    return {
        'status': 'ok' if estimate is not None else 'failed',
        'failure_reason': None if estimate is not None else 'no_speed_estimate',
        'estimated_raw_kmh': estimate,
        'raw_mean_kmh': robust_mean(raw_values),
        'observation_count': len(obs),
        'valid_segment_count': valid_count,
        'selected_candidate_row_count': len(candidate_rows),
        'candidate_valid_ratio': valid_count / max(len(candidate_rows), 1),
        'speed_cv': speed_cv,
        'confidence': confidence,
        'confidence_factors': {
            'coverage_quality': coverage_quality,
            'valid_ratio': valid_ratio,
            'selected_candidate_row_count': len(candidate_rows),
            'cv_quality': cv_quality,
            'mean_detection_confidence': mean_det_conf,
            'det_conf_quality': det_conf_quality,
            'estimate_available': estimate is not None,
        },
        'timeseries': rows,
    }


In [ ]:
# Cell 9 — Run tracking and raw speed extraction once per video
# This is the slowest cell. It caches track/speed outputs in Drive.

TRACK_CACHE = ARTIFACT_DIR / 'vs13_track_speed_cache.json'

if TRACK_CACHE.exists():
    cache = json.loads(TRACK_CACHE.read_text())
    print('Loaded cache:', TRACK_CACHE, 'videos:', len(cache.get('videos', [])))
else:
    cache = {'experiment_id': EXPERIMENT_ID, 'created_at': now_utc(), 'videos': []}

cached_by_path = {v['video_path']: v for v in cache.get('videos', [])}

base_params = {
    'horizontal_fov_deg': 70.0,
    'vehicle_height_m': 1.55,
    'moving_average_window': 25,
    'max_segment_speed_kmh': 180.0,
    'segment_trim_fraction': 0.10,
    'min_bbox_height_ratio': 0.0,
}

new_rows = []
for rec in tqdm(manifest_df.to_dict('records'), desc='VS13 videos'):
    video_path = rec['video_path']
    cached_row = cached_by_path.get(video_path)
    if cached_row and cached_row.get('track_result'):
        # Reuse expensive tracking, but refresh derived speed/confidence fields when the formula changes.
        track = cached_row['track_result']
        speed = compute_speed_candidate(track, base_params)
        row = dict(cached_row)
        row.update(rec)
        row.update({
            'track_status': track.get('status'),
            'track_failure_reason': track.get('failure_reason'),
            'fps': track.get('fps'),
            'frame_count': track.get('frame_count'),
            'resolution': track.get('resolution'),
            'runtime_s': track.get('runtime_s'),
            'track_count': track.get('track_count'),
            'selected_track_id': track.get('selected_track_id'),
            'selected_observation_count': track.get('selected_observation_count'),
            'selected_mean_confidence': track.get('selected_mean_confidence'),
            'base_estimated_raw_kmh': speed.get('estimated_raw_kmh'),
            'base_confidence': speed.get('confidence'),
            'base_speed_cv': speed.get('speed_cv'),
            'base_valid_segment_count': speed.get('valid_segment_count'),
            'base_candidate_valid_ratio': speed.get('candidate_valid_ratio'),
            'base_confidence_factors': speed.get('confidence_factors'),
            'track_result': track,
        })
        new_rows.append(row)
        continue
    track = extract_main_track(Path(video_path), frame_stride=FRAME_STRIDE)
    speed = compute_speed_candidate(track, base_params)
    row = dict(rec)
    row.update({
        'track_status': track.get('status'),
        'track_failure_reason': track.get('failure_reason'),
        'fps': track.get('fps'),
        'frame_count': track.get('frame_count'),
        'resolution': track.get('resolution'),
        'runtime_s': track.get('runtime_s'),
        'track_count': track.get('track_count'),
        'selected_track_id': track.get('selected_track_id'),
        'selected_observation_count': track.get('selected_observation_count'),
        'selected_mean_confidence': track.get('selected_mean_confidence'),
        'base_estimated_raw_kmh': speed.get('estimated_raw_kmh'),
        'base_confidence': speed.get('confidence'),
        'base_speed_cv': speed.get('speed_cv'),
        'base_valid_segment_count': speed.get('valid_segment_count'),
        'base_candidate_valid_ratio': speed.get('candidate_valid_ratio'),
        'base_confidence_factors': speed.get('confidence_factors'),
        'track_result': track,
    })
    new_rows.append(row)
    cache['videos'] = new_rows
    TRACK_CACHE.write_text(json.dumps(cache, indent=2), encoding='utf-8')

cache['videos'] = new_rows
TRACK_CACHE.write_text(json.dumps(cache, indent=2), encoding='utf-8')
print('Wrote cache:', TRACK_CACHE)

base_df = pd.DataFrame([{k:v for k,v in row.items() if k not in {'track_result'}} for row in new_rows])
base_csv = ARTIFACT_DIR / 'vs13_base_speed_candidates.csv'
base_df.to_csv(base_csv, index=False)
print('base_csv:', base_csv)
display(base_df[['split','vehicle','video_name','gt_speed_kmh','track_status','selected_observation_count','base_estimated_raw_kmh','base_confidence','base_speed_cv']].head(50))

In [ ]:
# Cell 10 — Wide-subset leave-one-vehicle-out calibration and lightweight training


def iter_param_grid(grid):
    keys = list(grid.keys())
    def rec(idx, current):
        if idx == len(keys):
            yield dict(current)
            return
        key = keys[idx]
        for value in grid[key]:
            current[key] = value
            yield from rec(idx + 1, current)
    yield from rec(0, {})


def rmse_np(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def feature_frame(df):
    out = df.copy()
    for col in FEATURE_COLUMNS:
        if col not in out.columns:
            out[col] = np.nan
    return out[FEATURE_COLUMNS].astype(float)


def fit_predict_calibration(train_df, test_df, method):
    train_df = train_df[train_df['pred_raw_kmh'].notna() & (train_df['pred_raw_kmh'] > 0)].copy()
    test_df = test_df[test_df['pred_raw_kmh'].notna() & (test_df['pred_raw_kmh'] > 0)].copy()
    if len(train_df) < 3 or len(test_df) == 0:
        return None, None
    y_train = train_df['gt_speed_kmh'].astype(float).to_numpy()
    x_train = train_df['pred_raw_kmh'].astype(float).to_numpy()
    x_test = test_df['pred_raw_kmh'].astype(float).to_numpy()
    params = {'calibration_method': method}

    if method == 'global_alpha':
        alpha = float(np.median(y_train / np.maximum(x_train, 1e-6)))
        pred = x_test * alpha
        params['global_scale_alpha'] = alpha
    elif method == 'linear_raw':
        a, b = np.polyfit(x_train, y_train, 1)
        pred = x_test * float(a) + float(b)
        params['linear_a'] = float(a)
        params['linear_b'] = float(b)
    elif method == 'huber_raw':
        model_cal = make_pipeline(StandardScaler(), HuberRegressor(epsilon=1.35, alpha=0.0001, max_iter=500))
        model_cal.fit(x_train.reshape(-1, 1), y_train)
        pred = model_cal.predict(x_test.reshape(-1, 1))
        params['model'] = 'HuberRegressor(raw_speed)'
    elif method == 'ridge_features':
        model_cal = make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), Ridge(alpha=1.0))
        model_cal.fit(feature_frame(train_df), y_train)
        pred = model_cal.predict(feature_frame(test_df))
        params['model'] = 'Ridge(features)'
    elif method == 'huber_features':
        model_cal = make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), HuberRegressor(epsilon=1.35, alpha=0.0001, max_iter=500))
        model_cal.fit(feature_frame(train_df), y_train)
        pred = model_cal.predict(feature_frame(test_df))
        params['model'] = 'HuberRegressor(features)'
    elif method == 'random_forest_features':
        model_cal = make_pipeline(
            SimpleImputer(strategy='median'),
            RandomForestRegressor(n_estimators=160, max_depth=6, min_samples_leaf=3, random_state=SEED, n_jobs=-1),
        )
        model_cal.fit(feature_frame(train_df), y_train)
        pred = model_cal.predict(feature_frame(test_df))
        params['model'] = 'RandomForestRegressor(features)'
    elif method == 'gradient_boosting_features':
        model_cal = make_pipeline(
            SimpleImputer(strategy='median'),
            GradientBoostingRegressor(n_estimators=160, learning_rate=0.04, max_depth=2, random_state=SEED),
        )
        model_cal.fit(feature_frame(train_df), y_train)
        pred = model_cal.predict(feature_frame(test_df))
        params['model'] = 'GradientBoostingRegressor(features)'
    else:
        raise ValueError('unknown calibration method: ' + method)

    test_out = test_df.copy()
    test_out['calibrated_speed_kmh'] = np.maximum(pred, 0.0)
    test_out['abs_error_kmh'] = (test_out['calibrated_speed_kmh'] - test_out['gt_speed_kmh'].astype(float)).abs()
    test_out['rel_error_pct'] = test_out['abs_error_kmh'] / test_out['gt_speed_kmh'].astype(float) * 100
    return params, test_out


def base_prediction_df(rows, params):
    per_video = []
    for row in rows:
        track = row.get('track_result')
        speed = compute_speed_candidate(track, params) if track else {'status': 'failed'}
        out = {k: v for k, v in row.items() if k != 'track_result'}
        out.update({
            'pred_raw_kmh': speed.get('estimated_raw_kmh'),
            'candidate_confidence': speed.get('confidence'),
            'candidate_speed_cv': speed.get('speed_cv'),
            'valid_segment_count': speed.get('valid_segment_count'),
            'candidate_valid_ratio': speed.get('candidate_valid_ratio'),
            'selected_candidate_row_count': speed.get('selected_candidate_row_count'),
            'candidate_confidence_factors': speed.get('confidence_factors'),
            'candidate_status': speed.get('status'),
            'candidate_failure_reason': speed.get('failure_reason'),
        })
        per_video.append(out)
    return pd.DataFrame(per_video)


def evaluate_leave_one_vehicle_out(raw_df, methods):
    vehicles = sorted(raw_df['vehicle'].dropna().unique().tolist())
    fold_predictions = []
    fold_rows = []
    for held_out_vehicle in vehicles:
        train_df = raw_df[raw_df['vehicle'] != held_out_vehicle].copy()
        test_df = raw_df[raw_df['vehicle'] == held_out_vehicle].copy()
        for method in methods:
            cal_params, pred_df = fit_predict_calibration(train_df, test_df, method)
            if pred_df is None:
                continue
            pred_df['held_out_vehicle'] = held_out_vehicle
            pred_df['calibration_method'] = method
            fold_predictions.append(pred_df)
            fold_rows.append({
                'held_out_vehicle': held_out_vehicle,
                'calibration_method': method,
                'fold_count': int(len(pred_df)),
                'fold_mae_kmh': float(pred_df['abs_error_kmh'].mean()),
                'fold_rmse_kmh': rmse_np(pred_df['gt_speed_kmh'].astype(float), pred_df['calibrated_speed_kmh'].astype(float)),
                'fold_mean_rel_error_pct': float(pred_df['rel_error_pct'].mean()),
                **{k: v for k, v in (cal_params or {}).items() if k != 'model'},
            })
    if not fold_predictions:
        return pd.DataFrame(), pd.DataFrame()
    all_pred = pd.concat(fold_predictions, ignore_index=True)
    fold_df = pd.DataFrame(fold_rows)
    return fold_df, all_pred


def summarize_method_predictions(pred_df):
    rows = []
    for method, sdf in pred_df.groupby('calibration_method'):
        rows.append({
            'calibration_method': method,
            'video_count': int(len(sdf)),
            'vehicle_count': int(sdf['vehicle'].nunique()),
            'loo_mae_kmh': float(sdf['abs_error_kmh'].mean()),
            'loo_rmse_kmh': rmse_np(sdf['gt_speed_kmh'].astype(float), sdf['calibrated_speed_kmh'].astype(float)),
            'loo_median_abs_error_kmh': float(sdf['abs_error_kmh'].median()),
            'loo_p90_abs_error_kmh': float(sdf['abs_error_kmh'].quantile(0.90)),
            'loo_mean_rel_error_pct': float(sdf['rel_error_pct'].mean()),
        })
    return pd.DataFrame(rows).sort_values(['loo_mae_kmh', 'loo_rmse_kmh']).reset_index(drop=True)


all_rows = new_rows
first_stage_records = []
first_stage_predictions = []
param_list = list(iter_param_grid(PARAM_GRID))
print('First-stage parameter configs:', len(param_list), 'methods:', FIRST_STAGE_METHODS)

for params in tqdm(param_list, desc='first-stage LOO grid'):
    raw_df = base_prediction_df(all_rows, params)
    fold_df, pred_df = evaluate_leave_one_vehicle_out(raw_df, FIRST_STAGE_METHODS)
    if pred_df.empty:
        continue
    method_summary = summarize_method_predictions(pred_df)
    for row in method_summary.to_dict('records'):
        first_stage_records.append({**params, **row})
    pred_df['param_hash'] = hashlib.sha1(json.dumps(params, sort_keys=True).encode()).hexdigest()[:12]
    for k, v in params.items():
        pred_df[k] = v
    first_stage_predictions.append(pred_df)

first_stage_metrics_df = pd.DataFrame(first_stage_records).sort_values(['loo_mae_kmh', 'loo_rmse_kmh']).reset_index(drop=True)
first_stage_metrics_csv = ARTIFACT_DIR / 'vs13b_first_stage_loo_metrics.csv'
first_stage_metrics_df.to_csv(first_stage_metrics_csv, index=False)
print('first_stage_metrics_csv:', first_stage_metrics_csv)
display(first_stage_metrics_df.head(30))

if not first_stage_predictions:
    raise RuntimeError('No first-stage predictions were produced. Check track extraction and manifest.')

all_first_stage_predictions_df = pd.concat(first_stage_predictions, ignore_index=True)
first_stage_predictions_csv = ARTIFACT_DIR / 'vs13b_first_stage_loo_predictions.csv'
all_first_stage_predictions_df.to_csv(first_stage_predictions_csv, index=False)
print('first_stage_predictions_csv:', first_stage_predictions_csv)

# Second-stage tabular regressors only on the top unique parameter configurations.
second_stage_metrics_df = pd.DataFrame()
second_stage_predictions_df = pd.DataFrame()
if ENABLE_SECOND_STAGE_TABULAR:
    top_param_cols = list(PARAM_GRID.keys())
    top_params_df = first_stage_metrics_df[top_param_cols].drop_duplicates().head(SECOND_STAGE_TOP_PARAM_CONFIGS)
    second_records = []
    second_predictions = []
    print('Second-stage top param configs:', len(top_params_df), 'methods:', SECOND_STAGE_METHODS)
    for params in tqdm(top_params_df.to_dict('records'), desc='second-stage feature models'):
        # Pandas may carry numpy scalar types; normalize to native Python.
        params = {k: (float(v) if isinstance(v, (np.floating, float)) else int(v) if isinstance(v, (np.integer, int)) else v) for k, v in params.items()}
        raw_df = base_prediction_df(all_rows, params)
        fold_df, pred_df = evaluate_leave_one_vehicle_out(raw_df, SECOND_STAGE_METHODS)
        if pred_df.empty:
            continue
        method_summary = summarize_method_predictions(pred_df)
        for row in method_summary.to_dict('records'):
            second_records.append({**params, **row})
        pred_df['param_hash'] = hashlib.sha1(json.dumps(params, sort_keys=True).encode()).hexdigest()[:12]
        for k, v in params.items():
            pred_df[k] = v
        second_predictions.append(pred_df)
    if second_records:
        second_stage_metrics_df = pd.DataFrame(second_records).sort_values(['loo_mae_kmh', 'loo_rmse_kmh']).reset_index(drop=True)
        second_stage_predictions_df = pd.concat(second_predictions, ignore_index=True)
        second_stage_metrics_df.to_csv(ARTIFACT_DIR / 'vs13b_second_stage_feature_model_metrics.csv', index=False)
        second_stage_predictions_df.to_csv(ARTIFACT_DIR / 'vs13b_second_stage_feature_model_predictions.csv', index=False)
        display(second_stage_metrics_df.head(30))

combined_metrics_df = pd.concat([first_stage_metrics_df, second_stage_metrics_df], ignore_index=True) if len(second_stage_metrics_df) else first_stage_metrics_df.copy()
combined_metrics_df = combined_metrics_df.sort_values(['loo_mae_kmh', 'loo_rmse_kmh']).reset_index(drop=True)
combined_metrics_csv = ARTIFACT_DIR / 'vs13b_combined_calibration_metrics.csv'
combined_metrics_df.to_csv(combined_metrics_csv, index=False)
print('combined_metrics_csv:', combined_metrics_csv)
display(combined_metrics_df.head(40))

best_metrics = combined_metrics_df.iloc[0].to_dict()
best_method = best_metrics['calibration_method']
best_params = {k: best_metrics[k] for k in PARAM_GRID.keys()}
print('best_method:', best_method)
print('best_params:', best_params)

# Retrieve best predictions from the matching stage.
best_source_df = second_stage_predictions_df if best_method in SECOND_STAGE_METHODS and len(second_stage_predictions_df) else all_first_stage_predictions_df
mask = best_source_df['calibration_method'].eq(best_method)
for k, v in best_params.items():
    mask &= np.isclose(best_source_df[k].astype(float), float(v))
best_df = best_source_df[mask].copy().sort_values(['held_out_vehicle', 'gt_speed_kmh', 'video_name']).reset_index(drop=True)
best_csv = ARTIFACT_DIR / 'vs13b_best_loo_predictions.csv'
best_df.to_csv(best_csv, index=False)
print('best_csv:', best_csv)
display(best_df[['held_out_vehicle','vehicle','video_name','gt_speed_kmh','pred_raw_kmh','calibrated_speed_kmh','abs_error_kmh','rel_error_pct','candidate_confidence','candidate_speed_cv']].head(120))


In [ ]:
# Cell 11 — Plots
plot_df = best_df.copy()

plt.figure(figsize=(8, 7))
for split, marker in [('train','o'), ('val','s'), ('test','^')]:
    sdf = plot_df[plot_df['split'] == split]
    plt.scatter(sdf['gt_speed_kmh'], sdf['calibrated_speed_kmh'], label=split, marker=marker, s=80)
lo = min(plot_df['gt_speed_kmh'].min(), plot_df['calibrated_speed_kmh'].min()) * 0.9
hi = max(plot_df['gt_speed_kmh'].max(), plot_df['calibrated_speed_kmh'].max()) * 1.1
plt.plot([lo, hi], [lo, hi], 'k--', linewidth=1)
plt.xlabel('Ground truth speed (km/h)')
plt.ylabel('Calibrated predicted speed (km/h)')
plt.title('VS13 calibrated speed sanity check')
plt.grid(True, alpha=0.3)
plt.legend()
scatter_path = PLOT_DIR / 'vs13_gt_vs_pred_scatter.png'
plt.savefig(scatter_path, dpi=160, bbox_inches='tight')
plt.show()
print(scatter_path)

plt.figure(figsize=(12, 5))
plot_df_sorted = plot_df.sort_values(['split','vehicle','video_name']).reset_index(drop=True)
labels = [f"{r.vehicle}\n{r.video_name[:12]}" for r in plot_df_sorted.itertuples()]
plt.bar(range(len(plot_df_sorted)), plot_df_sorted['abs_error_kmh'])
plt.xticks(range(len(plot_df_sorted)), labels, rotation=60, ha='right')
plt.ylabel('Absolute error (km/h)')
plt.title('VS13 calibrated speed absolute error')
plt.grid(axis='y', alpha=0.3)
error_path = PLOT_DIR / 'vs13b_loo_abs_error_by_video.png'
plt.savefig(error_path, dpi=160, bbox_inches='tight')
plt.show()
print(error_path)

# Confidence vs error
plt.figure(figsize=(8, 5))
plt.scatter(plot_df['candidate_confidence'], plot_df['abs_error_kmh'], s=80)
plt.xlabel('Candidate confidence')
plt.ylabel('Absolute error (km/h)')
plt.title('Confidence vs speed error')
plt.grid(True, alpha=0.3)
conf_path = PLOT_DIR / 'vs13b_loo_confidence_vs_error.png'
plt.savefig(conf_path, dpi=160, bbox_inches='tight')
plt.show()
print(conf_path)

In [ ]:
# Cell 12 — Write final summary report JSON + Markdown
summary = {
    'experiment_id': EXPERIMENT_ID,
    'created_at_utc': now_utc(),
    'purpose': 'Wide-subset VS13 leave-one-vehicle-out calibration/training test for monocular speed candidates.',
    'model_path': MODEL_PATH,
    'dataset': {
        'name': 'VS13 audio-video vehicle speed estimation dataset',
        'source': 'https://slobodan.ucg.ac.me/science/vs13/',
        'selected_vehicles': SELECTED_VEHICLES,
        'package_count': len(VS13_PACKAGES),
        'max_videos_per_vehicle': MAX_VIDEOS_PER_VEHICLE,
        'manifest_rows': int(len(manifest_df)),
    },
    'evaluation': {
        'strategy': 'leave-one-vehicle-out cross-validation',
        'why': 'prevents frame/video leakage and tests transfer to unseen vehicle packages',
    },
    'best_metrics': best_metrics,
    'outputs': {
        'manifest_csv': str(manifest_path),
        'base_candidates_csv': str(base_csv),
        'first_stage_metrics_csv': str(first_stage_metrics_csv),
        'first_stage_predictions_csv': str(first_stage_predictions_csv),
        'combined_metrics_csv': str(combined_metrics_csv),
        'best_predictions_csv': str(best_csv),
        'plots_dir': str(PLOT_DIR),
    },
    'limitations': [
        'VS13 speeds are maintained by cruise control and are appropriate for calibration sanity checks, not legal enforcement claims.',
        'Camera viewpoint differs from the local demo videos, so calibration transfer is not guaranteed.',
        'This notebook trains/tunes calibration regressors, not a new end-to-end neural speed model.',
        'FTR results.json does not require a speed field; use this module as support/evidence unless error is consistently low.',
    ],
}
if ENABLE_SECOND_STAGE_TABULAR and len(second_stage_metrics_df):
    summary['outputs']['second_stage_feature_model_metrics_csv'] = str(ARTIFACT_DIR / 'vs13b_second_stage_feature_model_metrics.csv')
    summary['outputs']['second_stage_feature_model_predictions_csv'] = str(ARTIFACT_DIR / 'vs13b_second_stage_feature_model_predictions.csv')

summary_json = ARTIFACT_DIR / 'speed_exp_006b_vs13_wide_subset_calibration_summary.json'
summary_json.write_text(json.dumps(summary, indent=2), encoding='utf-8')
print('summary_json:', summary_json)

report = ARTIFACT_DIR / 'speed_exp_006b_vs13_wide_subset_calibration_report.md'
lines = []
lines.append('# SPEED-EXP-006B VS13 Wide-Subset Calibration')
lines.append('')
lines.append('This experiment evaluates the current monocular bbox-geometry speed candidate on a wider VS13 subset using leave-one-vehicle-out cross-validation.')
lines.append('')
lines.append('## Best Metrics')
lines.append('')
for k, v in best_metrics.items():
    lines.append(f'- `{k}`: `{v}`')
lines.append('')
lines.append('## Outputs')
lines.append('')
for k, v in summary['outputs'].items():
    lines.append(f'- `{k}`: `{v}`')
lines.append('')
lines.append('## Decision Rule')
lines.append('')
lines.append('- If leave-one-vehicle-out MAE/RMSE is consistently low, report speed as `dataset-calibrated approximate candidate`.')
lines.append('- If errors remain vehicle- or speed-bin-dependent, keep speed as `relative/support evidence` for FTR and project reports.')
lines.append('')
lines.append('## Limitations')
lines.append('')
for item in summary['limitations']:
    lines.append(f'- {item}')
report.write_text('\n'.join(lines), encoding='utf-8')
print('report:', report)


In [ ]:
# Cell 13 — Next-step guidance
print('Review these files in Drive:')
print('1)', ARTIFACT_DIR / 'vs13b_combined_calibration_metrics.csv')
print('2)', ARTIFACT_DIR / 'vs13b_best_loo_predictions.csv')
print('3)', ARTIFACT_DIR / 'speed_exp_006b_vs13_wide_subset_calibration_summary.json')
print('4)', PLOT_DIR)
print('\nDecision rule:')
print('- If LOO MAE/RMSE is consistently low across held-out vehicles, keep speed as dataset-calibrated approximate candidate.')
print('- If errors are concentrated in specific vehicles/speed bins, add segment selector diagnostics or keep speed as relative/support evidence.')
print('- Do not block FTR Docker/results.json work on this module because FTR does not require speed output.')
